In [ ]:
from Bio import AlignIO, SeqIO
from Bio.SeqRecord import SeqRecord
import numpy as np
from collections import defaultdict

def find_insertions(alignment_file, output_fasta, output_summary, min_len=3, gap_thr=0.6, domains_file = None):

#если есть файл с доменами
    domains = {}
    if domains_file is not None:
        with open(domains_file, "r") as f:
            for line in f:
                if line.startswith("#"):
                    continue
                p = line.strip().split()
                if len(p) < 23:
                    continue
                try:
                    sid = p[3]
                    start = int(p[19])
                    end = int(p[20])
                except:
                    continue

                if sid not in domains:
                    domains[sid] = []
                domains[sid].append((start, end))

        for sid in domains:
            domains[sid].sort()

 #чтение выравниания
    aln = AlignIO.read(alignment_file, "fasta")
    arr = np.array([list(r.seq) for r in aln])

    found = []

#поиск вставок 
    for i, rec in enumerate(aln):
        sid = rec.id
        L = arr.shape[1]
        pos = 0

        while pos < L:
            if arr[i, pos] != "-":

                other = np.concatenate([arr[:i, pos], arr[i+1:, pos]])
                if np.sum(other == "-") / len(other) >= gap_thr:
                    s = pos
                    while pos < L and arr[i, pos] != "-":
                        other2 = np.concatenate([arr[:i, pos], arr[i+1:, pos]])
                        if np.sum(other2 == "-") / len(other2) < gap_thr:
                            break
                        pos += 1
                    e = pos
                    seq = "".join(arr[i, s:e])
                    real_len = len(seq)

#фильтрация по длине
                    if real_len >= min_len:
                        #аннотация относительно доменов
                        annot = "NA"
                        if domains_file is not None and sid in domains:
                            for ds, de in domains[sid]:
                                if s + 1 >= ds and e <= de:
                                    annot = f"inside_domain {ds}-{de}"
                                    break
                            else:
                                for j in range(len(domains[sid]) - 1):
                                    d1_e = domains[sid][j][1]
                                    d2_s = domains[sid][j + 1][0]
                                    if s + 1 > d1_e and e < d2_s:
                                        annot = f"between {d1_e}-{d2_s}"
                                        break

                        found.append((sid, seq, s + 1, e, real_len, annot))
                else:
                    pos += 1
            else:
                pos += 1

#объединение вставок
    records = []
    for n, (sid, seq, aln_start, aln_end, length, annotation) in enumerate(found, 1):
        desc = f"len={length}"
        if annotation != "NA":
            desc += f";{annotation}"

        records.append(
            SeqRecord(
                seq=seq,
                id=f"{sid}|{aln_start}-{aln_end}",
                description=desc
            )
        )

    SeqIO.write(records, output_fasta, "fasta")

#группировка вставок по расположению
    groups = defaultdict(list)
    for sid, seq, aln_start, aln_end, length, annotation in found:
        key = (aln_start, aln_end)
        groups[key].append(annotation)

    with open(output_summary, "w") as out:
        out.write("aln_start\taln_end\tcount\tannotations\n")
        for (s, e), annots in sorted(groups.items()):
            uniq = sorted(set(a for a in annots if a != "NA"))
            out.write(
                f"{s}\t{e}\t{len(annots)}\t{','.join(uniq) if uniq else 'NA'}\n"
            )

    return found



find_insertions(
    alignment_file=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\classical_swine_fever_virus\swine\swine_mafft.fasta",
    output_fasta=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\classical_swine_fever_virus\swine\all_insertions.fasta",
    output_summary=r"C:\Users\2slon\OneDrive\Рабочий стол\5sem\pestiviruses\final\classical_swine_fever_virus\swine\insertions_summary.tsv",
    min_len=3,
    gap_thr=0.8
)
